# rag-ab-tester quickstart (fully local)

Run an A/B test of two RAG pipeline variants on your machine, with **no API keys**:

| Piece | What we use |
|---|---|
| Generator + judge LLM | [Ollama](https://ollama.com) (`qwen2.5:7b`) through its OpenAI-compatible endpoint |
| Embeddings | Hugging Face `sentence-transformers` models, passed in through the SDK |
| Vector store | built-in `InMemoryIndex` (no FAISS or database needed) |
| Documents | the sample markdown files in `data/docs` |

**You will:** (1) check the setup, (2) create a small test set, (3) define two pipeline variants,
(4) run the experiment, (5) read the result. About 2-3 minutes per experiment on a laptop.

### Prerequisites
```bash
brew install ollama && ollama serve        # or the desktop app
ollama pull qwen2.5:7b
uv sync --all-extras                        # or: pip install -e ".[all]"
pip install sentence-transformers jupyter   # embeddings + notebook (uv: uv run --with ...)
```

In [ ]:
import json
import os
import sys
from pathlib import Path

# Work from the repo root so data/docs and data/test_set.json resolve.
root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("repo root:", root)

## 1. Configure the LLM
The built-in provider talks to any OpenAI-compatible server. For Ollama, `LLM_API_KEY` can stay unset.

In [ ]:
os.environ["LLM_MODEL"] = "qwen2.5:7b"
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
# Optional: use a different model as judge to avoid self-preference bias, e.g.
# os.environ["JUDGE_LLM_MODEL"] = "llama3.1:8b"   # (ollama pull it first)

import urllib.request

try:
    tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3))
    print("Ollama is up. Models:", [m["name"] for m in tags["models"]])
except Exception as e:
    raise SystemExit(f"Ollama is not reachable ({e}). Start it with `ollama serve`.") from e

## 2. Test set
Each case is a question plus a short ground-truth answer. These come from `data/docs`. Use 30+ cases for real decisions; 10 keeps this demo fast.

In [ ]:
test_set = [
    {
        "query": "What chunk size is the most common default?",
        "ground_truth": "512 tokens is the most common default, balancing precision and recall.",
    },
    {
        "query": "What is a typical overlap for 512-token chunks?",
        "ground_truth": "A 50-token overlap.",
    },
    {
        "query": "What is a disadvantage of large chunks of 1024-2048 tokens?",
        "ground_truth": "Lower retrieval precision, more noise passed to the LLM, and degraded embedding quality on very long chunks.",
    },
    {
        "query": "What does a semantic chunker do?",
        "ground_truth": "It uses sentence embeddings to detect topic shifts and splits at semantic boundaries; it is slower and needs an embedding call per sentence.",
    },
    {
        "query": "How many dimensions does nomic-embed-text have and what is its context window?",
        "ground_truth": "768 dimensions and an 8192-token context window.",
    },
    {
        "query": "Which local embedding model is recommended for best quality?",
        "ground_truth": "mxbai-embed-large.",
    },
    {
        "query": "What happens if you embed documents with one model and queries with another?",
        "ground_truth": "The vector spaces are incompatible and retrieval quality degrades to near-random.",
    },
    {
        "query": "What benchmark is the standard for evaluating embedding models and which category matters for RAG?",
        "ground_truth": "MTEB, the Massive Text Embedding Benchmark; the Retrieval category score is most relevant for RAG.",
    },
    {
        "query": "What kind of search do vector databases perform instead of exact matching?",
        "ground_truth": "Approximate nearest-neighbour (ANN) search to find semantically similar items.",
    },
    {
        "query": "What is the cost of OpenAI text-embedding-3-small?",
        "ground_truth": "$0.02 per million tokens.",
    },
]
Path("data").mkdir(exist_ok=True)
Path("data/test_set.json").write_text(json.dumps(test_set, indent=2))
print(len(test_set), "test cases written to data/test_set.json")

## 3. Define the pipeline variants
A variant is a `RAGPipeline` subclass that implements `get_chunk_size()` and `get_prompt()`, and optionally `get_embeddings()`.
An experiment file exposes `CONTROL`, `CHALLENGER` and their names. The experiment files live in `examples/experiments/` (they are tested in CI); the cells below print them. Open and edit them freely.

First, a shared helper: a Hugging Face embedder (any object with `embed_documents` and `embed_query` works) and a base pipeline.

In [ ]:
print(Path("examples/experiments/hf_base.py").read_text())

### Experiment A: chunk size (512 vs 256)

In [ ]:
print(Path("examples/experiments/exp_chunk_size.py").read_text())

### Experiment B: embedding model (MiniLM vs BGE-small)

In [ ]:
print(Path("examples/experiments/exp_embedder.py").read_text())

### Experiment C: prompt template (open vs strict grounding)
Only `get_prompt()` differs. Chunking, embeddings and retrieval are identical.

In [ ]:
print(Path("examples/experiments/exp_prompt.py").read_text())

### Experiment D: retrieval method (semantic vs hybrid BM25 + vector)
This is a *structural* change: override `ingest_texts()` to also build a keyword index and `retrieve()` to merge both result lists.
Any retrieval logic (rerankers, query rewriting, multi-query, ...) plugs in the same way. Needs the `docs` extra (`rank-bm25`).

In [ ]:
print(Path("examples/experiments/exp_retrieval.py").read_text())

## 4. Run the experiments
Each variant is ingested into its own in-memory index, every question is answered, and the judge scores faithfulness, answer relevance, context precision and context recall. Significance is tested per metric: `p < 0.05` means the difference is unlikely to be noise, and Cohen's d is the effect size.
Each run takes roughly 2-3 minutes (more for the first, which downloads the embedding models). Run only the ones you care about.

In [ ]:
from rag_ab_tester import run_experiment

sys.path.insert(0, str(root / "examples/experiments"))  # so experiment files can import hf_base


def run(name):
    return run_experiment(
        f"examples/experiments/{name}.py",
        test_set_path="data/test_set.json",
        docs_path="data/docs",
        save_json=True,  # also writes results/<experiment>.json
    )


results = {}

### A. Chunk size

In [ ]:
results["chunk size"] = run("exp_chunk_size")

### B. Embedding model

In [ ]:
results["embedding model"] = run("exp_embedder")

### C. Prompt template

In [ ]:
results["prompt"] = run("exp_prompt")

### D. Retrieval method

In [ ]:
results["retrieval"] = run("exp_retrieval")

## 5. Summary across experiments
Overall mean score per variant (average of the four metrics) and the verdict. With 10 questions only large effects are significant; use 30+ for real decisions.

In [ ]:
def ov(v):
    return sum(s.overall() for s in v.scores) / len(v.scores)


rows = []
for label, r in results.items():
    sig = [c.metric for c in r.comparisons if c.significant]
    rows.append(
        (
            label,
            r.control_name,
            round(ov(r.control), 3),
            r.challenger_name,
            round(ov(r.challenger), 3),
            r.overall_winner,
            ", ".join(sig) or "-",
        )
    )

print(
    f"{'experiment':<17}{'control':<14}{'score':<7}{'challenger':<18}{'score':<7}{'winner':<18}significant on"
)
for row in rows:
    print(f"{row[0]:<17}{row[1]:<14}{row[2]:<7}{row[3]:<18}{row[4]:<7}{row[5]:<18}{row[6]}")

## Next steps
- **Combine the winners:** set the best chunk size, embedder, prompt and retrieval in one pipeline and re-run against a larger test set to confirm the gains stack.
- Swap the LLM: pass `llm=` / `judge_llm=` to `run_experiment` (any LangChain chat model or a `str -> str` function).
- Swap the vector store: pass `index_factory=` (see `LangChainVectorStoreIndex` for FAISS, Chroma, pgvector, ...).
- Use your own documents in `data/docs` and write 30+ questions.
- CLI equivalent: `rag-ab run examples/experiments/exp_chunk_size.py`.